# 01 · Build the training dataset

Streams RAID and DACTYL, draws a balanced sample stratified by generator,
domain and label, extracts the 30 features, and writes two Parquet files.

**Two datasets, because there are two models (PRD 8.1).**

| | rows | adversarial |
|---|---|---|
| `train_a.parquet` | Model A, the strict detector | included, ~15% of the mix |
| `train_b.parquet` | Model B, the surrogate | excluded entirely |

Excluding the adversarial rows from B is not an oversight — it is what makes B
a faithful stand-in for third-party detectors, which have no such hardening.

**The labelling rule that matters (E8).** RAID's paraphrase / synonym /
homoglyph variants are the humanized-text augmentation, free. Humanized *AI*
text stays labelled AI. Humanized *human* text stays labelled **human** —
labelling a rewritten human essay as AI would teach the model that editing is
evidence of machine authorship, which is exactly how detectors end up
punishing careful writers and non-native speakers (R2, A4).

Both datasets stream, so RAID's 16.7 GB never touches Kaggle's 20 GB disk.

Runtime: minutes on a smoke test, 1–2 hours at 400k rows. **CPU only — turn
the accelerator off for this stage to save GPU quota.**


In [ ]:
# Setup. Run once per session — everything below depends on it.
!pip install -q "transformers>=4.44" "datasets>=2.20" sentencepiece onnx onnxruntime \
    "optimum[onnxruntime]" pyarrow

import sys, os
from pathlib import Path

# Change this if you forked the repo. Public repo => no token needed.
GIT_URL = "https://github.com/ByteCraft-9/ai-text-humanizer.git"

# Either attached as a Kaggle Dataset named `ai-detector-repo`, or cloned.
REPO = Path("/kaggle/input/ai-detector-repo") if Path("/kaggle/input/ai-detector-repo").exists() \
       else Path("/kaggle/working/ai-detector")
if not REPO.exists():
    !git clone --depth 1 $GIT_URL /kaggle/working/ai-detector

sys.path.insert(0, str(REPO / "training"))
sys.path.insert(0, str(REPO / "api"))

# parents=True so this also works off-Kaggle (Colab, a local box) after
# pointing WORK somewhere that exists.
WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / "data"; DATA.mkdir(parents=True, exist_ok=True)
MODELS = WORK / "models"; MODELS.mkdir(parents=True, exist_ok=True)

print("repo:", REPO)
print("work:", WORK)
assert (REPO / "training" / "lib").is_dir(), "repo not found — check GIT_URL"


In [ ]:
# ---------------------------------------------------------------------------
# Run configuration. Read this cell before starting anything else.
# ---------------------------------------------------------------------------
#
# Kaggle gives 30 GPU-hours a week. A full run is 8-16 of them, so you cannot
# afford to discover a bug at hour six. Leave SMOKE_TEST = True for the first
# pass: it runs the entire pipeline end to end in well under an hour on a
# tiny sample. If stage 4 completes, the chain works. Then set it False and
# run for real.

SMOKE_TEST = True

if SMOKE_TEST:
    SAMPLE_ROWS = 5_000     # rows per dataset
    EPOCHS = 1
else:
    SAMPLE_ROWS = 400_000   # PRD 12.2
    EPOCHS = 3

print(f"{'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'}: "
      f"{SAMPLE_ROWS:,} rows/dataset, {EPOCHS} epoch(s)")
if not SMOKE_TEST:
    print("Expect ~1-2 h for stage 1, then 4-8 h per model. Use "
          "Save Version -> Save & Run All so a browser disconnect cannot kill it.")


In [ ]:
from pathlib import Path
from lib.data import SampleSpec, build_dataset, feature_statistics

SPEC = SampleSpec(total=SAMPLE_ROWS, adversarial_share=0.15, human_share=0.5)
frame_a = build_dataset(DATA / "train_a.parquet", SPEC, include_adversarial=True)


In [ ]:
spec_b = SampleSpec(total=SAMPLE_ROWS, adversarial_share=0.0, human_share=0.5,
                    seed=SPEC.seed + 1)
frame_b = build_dataset(DATA / "train_b.parquet", spec_b, include_adversarial=False)


In [ ]:
# Gate: class balance and domain coverage must be verified before training
# (PRD 18, phase 1). A skewed sample produces a model that looks fine on its
# own validation split and fails on everything else.
for name, frame in (("A", frame_a), ("B", frame_b)):
    print(f"--- Model {name}: {len(frame):,} rows")
    print(frame["label"].value_counts(normalize=True).round(3).to_dict())
    print("  generators:", frame["generator"].nunique(), " domains:", frame["domain"].nunique())
    print("  adversarial share:", round(frame["adversarial"].mean(), 3))
    print("  words: median", int(frame["text"].str.split().str.len().median()))

balance = frame_a["label"].mean()
assert 0.4 < balance < 0.6, f"Model A is unbalanced: {balance:.3f} positive"
assert frame_b["adversarial"].sum() == 0, "Model B must contain no adversarial rows"
print("\nGate passed.")


In [ ]:
# Feature standardisation statistics.
#
# Copy these into api/_lib/features.py (FEATURE_MEAN / FEATURE_STD) and commit
# before deploying. The values shipped there are rough placeholders; leaving
# them means inference standardises with different numbers than training used,
# which costs real accuracy. Skip this on a smoke test.
import json
stats = feature_statistics(frame_a)
(MODELS / "feature_stats.json").write_text(json.dumps(stats, indent=2))
print(json.dumps(stats, indent=2)[:900])
